# D1｜class MLP 完整訓練（練習）

**驗收標準**
- 用 `nn.Module` 寫兩層 MLP，並走出完整的一步：`zero_grad → forward → loss → backward → step`。
- 寫 `Dataset` / `DataLoader`，並在固定的 val set 上算 accuracy。eval 不更新權重。

**使用方式**
1. 蓋住剛才的範例再做。
2. 只改有 `# TODO` 的格子；測試格不要改。
3. 由上往下執行，每個測試印出 `PASS` 才往下。
4. 最後填「當日紀錄」。

| 段落 | 建議時間 |
|---|---|
| 0. 口述題 | 10 分 |
| 1. `class MLP` | 20 分 |
| 2. `train_step` | 20 分 |
| 3. 追問：梯度寫在哪 | 10 分 |
| 4. `Dataset` | 15 分 |
| 5. `DataLoader` | 10 分 |
| 6. eval accuracy | 15 分 |
| 7. 當日紀錄 | 5 分 |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def check(name, cond):
    print(("PASS  " if cond else "FAIL  ") + name)
    return bool(cond)


## 0. 口述題（先寫在這格，蓋筆記作答）

1. `super().__init__()` 在做什麼？漏掉會怎樣？
2. `backward()` 寫進哪裡？`optimizer.step()` 改哪裡？
3. 為什麼 `nn.Linear` 要在 `__init__` 裡寫成 `self.fc = ...`，`F.relu` 卻放在 `forward`？
4. 輸入 `x` 要不要 `requires_grad`？optimizer 應該拿到誰？
5. `torch.manual_seed(0)` 固定的是什麼？

**我的回答：**

1. 
2. 
3. 
4. 
5. 


## 1. `class MLP`

`x → Linear(d, h) → ReLU → Linear(h, c) → logits`

- 繼承 `nn.Module`，`__init__` 第一行呼叫 `super().__init__()`
- 屬性名字用 `self.fc1`、`self.fc2`
- `forward` 回傳 **logits**，不要 softmax
- ReLU 用 `F.relu`（沒有參數的運算）


In [ ]:
class MLP(nn.Module):
    def __init__(self, d, h, c):
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO: F.relu(self.fc1(x))，再過 self.fc2，回傳 logits
        raise NotImplementedError


In [ ]:
torch.manual_seed(0)
model = MLP(4, 8, 3)
x = torch.randn(6, 4)
logits = model(x)
names = {n for n, _ in model.named_parameters()}

check("logits shape (N, C)", tuple(logits.shape) == (6, 3))
check("fc1.weight shape (H, D)", tuple(model.fc1.weight.shape) == (8, 4))
check("fc2.weight shape (C, H)", tuple(model.fc2.weight.shape) == (3, 8))
check("權重 requires_grad", bool(model.fc1.weight.requires_grad and model.fc2.bias.requires_grad))
check("參數有註冊", {"fc1.weight", "fc1.bias", "fc2.weight", "fc2.bias"} <= names)

# fc1 輸出全是負的：ReLU 應擋成 0，logits 只剩 fc2 的 bias
relu_model = MLP(4, 8, 3)
with torch.no_grad():
    relu_model.fc1.weight.zero_()
    relu_model.fc1.bias.fill_(-2.0)
    relu_model.fc2.weight.fill_(1.0)
    relu_model.fc2.bias.fill_(3.0)
out = relu_model(torch.randn(5, 4))
check("負的隱藏層被 ReLU 擋掉", torch.allclose(out, torch.full_like(out, 3.0)))


## 2. `train_step`

走完**一步**，回傳這一步的 `loss`（`step` 之前算出來的那個 tensor）。

順序固定：

1. `opt.zero_grad()`：清掉上一步留在 `.grad` 的值
2. `logits = model(x)`
3. `loss = F.cross_entropy(logits, y)`
4. `loss.backward()`：只填 `.grad`
5. `opt.step()`：這時權重才改

`opt` 已經建好，不要在函式裡重做一個 optimizer。


In [ ]:
def train_step(model, opt, x, y):
    # TODO: zero_grad → forward → loss → backward → step，回傳 loss
    raise NotImplementedError


In [ ]:
torch.manual_seed(1)
model = MLP(4, 8, 3)
x = torch.randn(8, 4)
y = torch.randint(0, 3, (8,))
opt = torch.optim.SGD(model.parameters(), lr=0.1)
state = {k: v.detach().clone() for k, v in model.state_dict().items()}
w0 = model.fc1.weight.detach().clone()

loss = train_step(model, opt, x, y)

ref = MLP(4, 8, 3)
ref.load_state_dict(state)
with torch.no_grad():
    expected = F.cross_entropy(ref(x), y)

check("回傳的是 tensor", torch.is_tensor(loss))
check("回傳的是 step 前的 loss", torch.allclose(loss.detach(), expected))
check("loss 有限", torch.isfinite(loss.detach()).item())
check("step 後 fc1.weight 有變", not torch.equal(w0, model.fc1.weight.detach()))
check("x 沒有 grad", x.grad is None)

# 事先塞一個很大的舊梯度。有 zero_grad 的話，這一步的更新不會被它放大。
torch.manual_seed(2)
model = MLP(4, 8, 3)
x = torch.randn(8, 4)
y = torch.randint(0, 3, (8,))
opt = torch.optim.SGD(model.parameters(), lr=0.1)
model.fc1.weight.grad = torch.ones_like(model.fc1.weight) * 1000
w0 = model.fc1.weight.detach().clone()
train_step(model, opt, x, y)
delta = (model.fc1.weight.detach() - w0).abs().mean().item()
check("zero_grad 有清掉舊梯度", delta < 1.0)


## 3. 追問

先不要看上面的範例，用自己的話寫。

- `backward()` 之後，`fc1.weight` 的數值變了嗎？`.grad` 裡有什麼？
- `step()` 做的更新，和 `param -= lr * param.grad` 是什麼關係？
- 若迴圈裡漏了 `zero_grad()`，下一步的 `.grad` 會怎樣？

**我的回答：**

- 
- 
- 


## 4. `Dataset`

`Dataset` 只負責**一筆**資料。

- `__init__` 把傳進來的 `x`、`y` 存成 `self.x`、`self.y`
- `__len__` 回傳筆數
- `__getitem__(i)` 回傳第 `i` 筆：`(self.x[i], self.y[i])`，不要回傳整包


In [ ]:
class ToySet(Dataset):
    def __init__(self, x, y):
        # TODO
        raise NotImplementedError

    def __len__(self):
        # TODO
        raise NotImplementedError

    def __getitem__(self, i):
        # TODO: 第 i 筆的特徵和類別
        raise NotImplementedError


In [ ]:
torch.manual_seed(0)
x = torch.randn(5, 4)
y = torch.tensor([0, 1, 2, 1, 0])
ds = ToySet(x, y)
x0, y0 = ds[0]
x4, y4 = ds[4]

def label_is(v, n):
    if torch.is_tensor(v):
        return v.ndim == 0 and int(v) == n
    return int(v) == n

check("len 是筆數", len(ds) == 5)
check("第 0 筆特徵", torch.is_tensor(x0) and torch.equal(x0, x[0]))
check("第 0 筆標籤", label_is(y0, 0))
check("單筆 shape (D,)", torch.is_tensor(x0) and tuple(x0.shape) == (4,))
check("最後一筆", torch.is_tensor(x4) and torch.equal(x4, x[4]) and label_is(y4, 0))


## 5. `DataLoader`

把 `Dataset` 組成 batch。

- 回傳 `(train_loader, val_loader)`
- `batch_size` 用參數，不要寫死
- `train_loader`：`shuffle=True`
- `val_loader`：`shuffle=False`，每次 eval 順序才固定


In [ ]:
def make_loaders(train_ds, val_ds, batch_size):
    # TODO
    raise NotImplementedError


In [ ]:
train_x = torch.arange(10 * 4, dtype=torch.float).view(10, 4)
train_y = torch.arange(10) % 3
val_x = torch.arange(6 * 4, dtype=torch.float).view(6, 4)
val_y = torch.arange(6) % 3
train_ds = ToySet(train_x, train_y)
val_ds = ToySet(val_x, val_y)
train_loader, val_loader = make_loaders(train_ds, val_ds, batch_size=4)

from torch.utils.data import RandomSampler, SequentialSampler
xb, yb = next(iter(val_loader))
rows = torch.stack([val_ds[i][0] for i in range(4)])

check("回傳 DataLoader", isinstance(train_loader, DataLoader) and isinstance(val_loader, DataLoader))
check("val batch shape", tuple(xb.shape) == (4, 4) and tuple(yb.shape) == (4,))
check("10 筆、batch 4，訓練有 3 個 batch", len(train_loader) == 3)
check("train shuffle", isinstance(train_loader.sampler, RandomSampler))
check("val 不 shuffle，第一個 batch 是原本前 4 筆", isinstance(val_loader.sampler, SequentialSampler) and torch.equal(xb, rows))


## 6. eval accuracy

在一個 loader 上算答對比例。這是 benchmark：同一份資料，比較訓練前和訓練後。

- `model.eval()`：Dropout / BatchNorm 走推論。要在 `forward` 之前呼叫
- `torch.no_grad()` 或 `@torch.no_grad()`：不算梯度
- 預測類別是 `logits.argmax(dim=-1)`
- 把**所有 batch** 的答對筆數加總，再除以總筆數。不要只留最後一個 batch，也不要先各自算 acc 再平均
- 不要 `backward`，不要 `step`


In [ ]:
def accuracy(model, loader):
    # TODO: eval + no_grad，回傳 0 到 1 的答對比例
    raise NotImplementedError


In [ ]:
x = torch.randn(6, 4)
y = torch.tensor([0, 0, 0, 0, 1, 1])  # 前 4 筆是 0，後 2 筆是 1
loader = DataLoader(ToySet(x, y), batch_size=4, shuffle=False)

class AlwaysZero(nn.Module):
    def forward(self, x):
        logits = torch.zeros(x.shape[0], 3)
        logits[:, 0] = 10.0  # 每筆都猜類別 0
        return logits

# 6 筆裡答對 4 筆。若只留最後一個 batch，會得到 0；若先算每個 batch 的 acc 再平均，會得到 0.5。
acc = accuracy(AlwaysZero(), loader)
check("全猜 0 的 acc 是 4/6", abs(acc - 4 / 6) < 1e-6)

class Probe(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 3)
        self.saw_grad = []
        self.saw_training = []

    def forward(self, x):
        self.saw_grad.append(torch.is_grad_enabled())
        self.saw_training.append(self.training)
        return self.fc(x)

probe = Probe()
probe.train()
w0 = probe.fc.weight.detach().clone()
accuracy(probe, loader)
check("forward 時關掉 autograd", len(probe.saw_grad) > 0 and all(v is False for v in probe.saw_grad))
check("forward 之前有 model.eval()", len(probe.saw_training) > 0 and all(v is False for v in probe.saw_training))
check("eval 不改權重", torch.equal(w0, probe.fc.weight.detach()))


## 7. 當日紀錄

- 日期：2026-10-15
- 題目：`class MLP` ＋ 一步訓練 ＋ Dataset / DataLoader / eval
- 第一個寫錯的順序：
- 漏掉 `super().__init__` / `zero_grad` / `step` / `shuffle` / `model.eval()` 時實際看到什麼：
- 下次辨識訊號：
- 驗收結果：□ MLP 全部 PASS　□ train_step 全部 PASS　□ Dataset / DataLoader PASS　□ accuracy PASS　□ 能講 backward 和 step 的差別
